# 🔐 Smart Contract Vulnerability Detector
## Fine-tuning `microsoft/codebert-base` với LoRA

Notebook này fine-tune **CodeBERT-base** bằng **LoRA** để phân loại `Vulnerable` / `Safe` trên mã nguồn Solidity Smart Contract.

| Thành phần | Chi tiết |
|---|---|
| Mô hình nền | `microsoft/codebert-base` |
| Kỹ thuật | LoRA (r=16, α=32) + Sequence Classification |
| Dữ liệu | `combined.json` (`Input` / `Output`) |
| Chia tập | Train 80% · Val 10% · Test 10% |
| Chống overfitting | 5 biến thể Prompt ngẫu nhiên |
| Xử lý code dài | Sliding Window chỉ cho **Safe** (stride 256) · **Vulnerable** dùng toàn bộ 512 tokens |
| Đánh giá | Majority Voting (5 prompt) + F1-score (sklearn) |

> **Lưu ý:** CodeBERT là encoder-only (BERT-based), context window tối đa **512 tokens**  
> (so với CodeLlama 4096 tokens).  
> **Chiến lược Sliding Window theo nhãn:**  
> - `Safe`: code thường dài → áp dụng sliding window (stride 256) để không bỏ sót ngữ cảnh  
> - `Vulnerable`: code ngắn (phạm vi 1 hàm) → đưa **toàn bộ** vào 512 tokens, không cắt  
> QLoRA 4-bit không cần thiết vì CodeBERT chỉ ~125M params — dùng LoRA thông thường.


## ⚙️ 0. Cài đặt thư viện

In [1]:
# %%capture
# Dùng transformers>=4.45 để tương thích peft mới (cần EncoderDecoderCache)
!pip install -q -U transformers datasets peft accelerate scikit-learn torchao

# Kiểm tra version sau khi cài
import importlib
for pkg in ["transformers", "peft", "accelerate", "torchao"]:
    v = importlib.import_module(pkg).__version__
    print(f"  {pkg}: {v}")
print("✅ Cài đặt hoàn tất!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 106.7 MB/s eta 0:00:00
  transformers: 5.5.4


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


  peft: 0.19.1
  accelerate: 1.13.0
  torchao: 0.17.0
✅ Cài đặt hoàn tất!


## 📦 1. Import thư viện & Cấu hình ban đầu

In [2]:
import json, re, random
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,  # Thay vì CausalLM
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType  # Bỏ QLoRA
from sklearn.metrics import f1_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# =========================================================
# CodeBERT: encoder-only, classification head tích hợp sẵn
# =========================================================
MODEL_NAME  = "microsoft/codebert-base"
LABEL2ID    = {"Safe": 0, "Vulnerable": 1}
ID2LABEL    = {0: "Safe", 1: "Vulnerable"}
MAX_TOKENS  = 512    # Giới hạn cứng của BERT-based models

_dev = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"✅ Import OK | Device: {_dev}")

✅ Import OK | Device: Tesla T4


## 📂 2. Nạp dữ liệu (`combined.json`)

> **Lưu ý Kaggle:** Upload `combined.json` vào phần **Datasets** rồi cập nhật `DATA_PATH` bên dưới cho khớp đường dẫn.

In [3]:
DATA_PATH = "/kaggle/input/datasets/thanhphuocjr/codellma-dataset/combined.json"  # <- Chỉnh đường dẫn

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

n_total = len(raw_data)
n_vuln  = sum(1 for d in raw_data if d["Output"] == "Vulnerable")
n_safe  = n_total - n_vuln
print(f"Tổng mẫu  : {n_total:,}")
print(f"Vulnerable: {n_vuln:,}  ({n_vuln / n_total * 100:.1f}%)")
print(f"Safe      : {n_safe:,}  ({n_safe / n_total * 100:.1f}%)")
print("\nVí dụ mẫu đầu tiên (400 ký tự):")
print(json.dumps(raw_data[0], indent=2, ensure_ascii=False)[:400])

Tổng mẫu  : 4,722
Vulnerable: 2,216  (46.9%)
Safe      : 2,506  (53.1%)

Ví dụ mẫu đầu tiên (400 ký tự):
{
  "Input": "    function _withdraw(address _account, uint256 _withdrawalID) internal override {\n        uint256 amount = withdrawLocks.withdraw(_account, _withdrawalID);\n\n        livepeer.withdrawStake(_withdrawalID);\n\n        steak.transfer(_account, amount);\n\n        emit Withdraw(_account, amount, _withdrawalID);\n    }",
  "Output": "Vulnerable"
}


## 🎲 3. Kỹ thuật 5 Biến thể Prompt (chống Overfitting)

Giữ nguyên 5 biến thể prompt. Với CodeBERT (classification), prompt được ghép theo chuẩn **BERT input**:
```
[CLS] {prompt_text} [SEP] {code} [SEP]
```
Tokenizer xử lý phần `[CLS]`/`[SEP]` tự động — ta chỉ cần truyền `(prompt_text, code)` dạng **text pair**.

In [4]:
# =============================================================
# 3a. Hàm trích xuất fn_name & contract_name bằng Regex
# =============================================================

def extract_fn_and_contract(code: str):
    """Trích xuất tên function và contract từ mã Solidity."""
    fn_match       = re.search(r'function\s+(\w+)\s*\(', code)
    contract_match = re.search(r'contract\s+(\w+)', code)
    fn_name       = fn_match.group(1)       if fn_match       else "unknownFunction"
    contract_name = contract_match.group(1) if contract_match else "UnknownContract"
    return fn_name, contract_name


# =============================================================
# 3b. Định nghĩa 5 biến thể Prompt (giữ nguyên)
# =============================================================

PROMPT_VARIANTS = [
    {   # Biến thể 1
        "a": "Devise a label name suitable for categorizing items as either vulnerable or safe.",
        "b": "Please review the code. Please find out if it is vulnerable.",
        "c": "The function {fn_name} from the contract {contract_name}.",
    },
    {   # Biến thể 2
        "a": "Suggest a label designation that clearly identifies an item's status as either vulnerable or safe.",
        "b": "Inspect the following Solidity code. Determine if there are any vulnerabilities present.",
        "c": "Observe the method {fn_name} within the smart contract {contract_name}.",
    },
    {   # Biến thể 3
        "a": "Invent a naming label that aptly segregates items into vulnerable or safe classifications.",
        "b": "Examine this Solidity script. Identify any potential security risks.",
        "c": "Review the function {fn_name} in the blockchain contract {contract_name}.",
    },
    {   # Biến thể 4
        "a": "Formulate a label descriptor that bifurcates objects into categories of vulnerable and safe.",
        "b": "Please assess the provided Solidity code for any security vulnerabilities.",
        "c": "Check the procedure {fn_name} in the digital contract {contract_name}.",
    },
    {   # Biến thể 5
        "a": "Propose a label nomenclature that aptly differentiates between vulnerable and safe states.",
        "b": "Evaluate the given Solidity function. Are there any security flaws?",
        "c": "Inspect the subroutine {fn_name} from the decentralized contract {contract_name}.",
    },
]


# =============================================================
# 3c. Tạo text_a (prompt instruction) cho CodeBERT text-pair
# =============================================================

def build_prompt_text(code: str, variant_idx: int = None) -> str:
    """
    Tạo phần text_a (instruction) dùng làm cặp với code trong BERT input.

    Args:
        code        : mã nguồn Solidity
        variant_idx : None -> random 1 trong 5 (khi train)
                      0..4 -> dùng đúng biến thể đó (khi val/test)
    Returns:
        text_a (str): chuỗi instruction ghép a + b + c
    """
    v = (PROMPT_VARIANTS[variant_idx]
         if variant_idx is not None
         else random.choice(PROMPT_VARIANTS))
    fn_name, contract_name = extract_fn_and_contract(code)
    c_filled = v["c"].format(fn_name=fn_name, contract_name=contract_name)
    return f"{v['a']} {v['b']} {c_filled}"


# -- Kiểm tra nhanh 2 biến thể --
for _i in [0, 2]:
    _t = build_prompt_text(raw_data[0]["Input"][:150], variant_idx=_i)
    print(f"=== Biến thể {_i + 1} ===")
    print(_t)
    print()

=== Biến thể 1 ===
Devise a label name suitable for categorizing items as either vulnerable or safe. Please review the code. Please find out if it is vulnerable. The function _withdraw from the contract UnknownContract.

=== Biến thể 3 ===
Invent a naming label that aptly segregates items into vulnerable or safe classifications. Examine this Solidity script. Identify any potential security risks. Review the function _withdraw in the blockchain contract UnknownContract.



## 🔤 4. Nạp Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print(f"✅ Tokenizer loaded | vocab_size = {tokenizer.vocab_size:,}")
print(f"   model_max_length = {tokenizer.model_max_length}")

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

✅ Tokenizer loaded | vocab_size = 50,265
   model_max_length = 512


## 🪟 5. Sliding Window — chỉ áp dụng cho nhãn Safe

**Lý do phân biệt theo nhãn:**
- `Vulnerable`: code thường ngắn (phạm vi 1 hàm) → đưa **toàn bộ** vào 512 tokens để model thấy đầy đủ ngữ cảnh lỗ hổng, không cắt xén
- `Safe`: code thường dài (nhiều hàm, contract lớn) → áp dụng **sliding window** stride 256 để không bỏ sót phần cuối

Budget token cho code: `512 - ~100 (prompt) - 3 (special tokens) = ~409 tokens`

In [6]:
# Token budget: MAX_TOKENS=512, dành ~100 cho prompt + 3 special tokens ([CLS],[SEP],[SEP])
CODE_MAX_TOKENS = MAX_TOKENS - 103   # ~409 tokens dành cho code
SAFE_STRIDE     = 256                 # stride cho Safe (code dài)


def apply_sliding_window_safe_only(
    tokenizer,
    code: str,
    label: str,
    max_tokens: int = CODE_MAX_TOKENS,
    stride: int = SAFE_STRIDE,
) -> list:
    """
    Chiến lược sliding window theo nhãn:

    - Vulnerable: KHÔNG cắt. Truncate tại max_tokens để giữ nguyên
                  ngữ cảnh lỗ hổng (vốn nằm gọn trong 1 hàm ngắn).
    - Safe      : Áp dụng sliding window (stride=SAFE_STRIDE) để bao phủ
                  toàn bộ code dài, mỗi chunk giữ nguyên nhãn Safe.

    Args:
        tokenizer  : HuggingFace tokenizer
        code       : mã nguồn Solidity
        label      : "Vulnerable" hoặc "Safe"
        max_tokens : kích thước cửa sổ tối đa dành cho code (tokens)
        stride     : bước trượt (tokens) — chỉ dùng khi label=Safe
    Returns:
        list[(chunk_code: str, label: str)]
    """
    token_ids = tokenizer.encode(code, add_special_tokens=False)

    # --- Vulnerable: giữ nguyên, truncate nếu cần ---
    if label == "Vulnerable":
        # Truncate tại max_tokens để tokenize_sample không mất dữ liệu
        truncated = tokenizer.decode(token_ids[:max_tokens], skip_special_tokens=True)
        return [(truncated, label)]

    # --- Safe: áp dụng sliding window ---
    # Nếu code đủ ngắn -> trả thẳng
    if len(token_ids) <= max_tokens:
        return [(code, label)]

    chunks = []
    start  = 0
    while start < len(token_ids):
        end        = min(start + max_tokens, len(token_ids))
        chunk_code = tokenizer.decode(token_ids[start:end], skip_special_tokens=True)
        chunks.append((chunk_code, label))
        if end >= len(token_ids):
            break
        start += stride
    return chunks


# -- Kiểm tra mẫu Safe dài nhất --
_long_safe = max(
    (d for d in raw_data if d["Output"] == "Safe"),
    key=lambda x: len(x["Input"])
)
_tlen_safe = len(tokenizer.encode(_long_safe["Input"], add_special_tokens=False))
_cks_safe  = apply_sliding_window_safe_only(tokenizer, _long_safe["Input"], "Safe")
print(f"[Safe]  Mẫu dài nhất : {_tlen_safe} tokens")
print(f"        Số chunk tạo : {len(_cks_safe)}")
for _i, (_c, _l) in enumerate(_cks_safe):
    _tl = len(tokenizer.encode(_c, add_special_tokens=False))
    print(f"  Chunk {_i}: {_tl} tokens | label={_l}")

# -- Kiểm tra mẫu Vulnerable dài nhất --
_long_vuln = max(
    (d for d in raw_data if d["Output"] == "Vulnerable"),
    key=lambda x: len(x["Input"])
)
_tlen_vuln = len(tokenizer.encode(_long_vuln["Input"], add_special_tokens=False))
_cks_vuln  = apply_sliding_window_safe_only(tokenizer, _long_vuln["Input"], "Vulnerable")
print(f"\n[Vuln]  Mẫu dài nhất : {_tlen_vuln} tokens")
print(f"        Số chunk tạo : {len(_cks_vuln)} (luôn = 1, không cắt)")
print(f"  Chunk 0: {len(tokenizer.encode(_cks_vuln[0][0], add_special_tokens=False))} tokens | label=Vulnerable")

Token indices sequence length is longer than the specified maximum sequence length for this model (97826 > 512). Running this sequence through the model will result in indexing errors


[Safe]  Mẫu dài nhất : 97826 tokens
        Số chunk tạo : 382
  Chunk 0: 409 tokens | label=Safe
  Chunk 1: 409 tokens | label=Safe
  Chunk 2: 409 tokens | label=Safe
  Chunk 3: 409 tokens | label=Safe
  Chunk 4: 409 tokens | label=Safe
  Chunk 5: 409 tokens | label=Safe
  Chunk 6: 409 tokens | label=Safe
  Chunk 7: 409 tokens | label=Safe
  Chunk 8: 409 tokens | label=Safe
  Chunk 9: 409 tokens | label=Safe
  Chunk 10: 409 tokens | label=Safe
  Chunk 11: 409 tokens | label=Safe
  Chunk 12: 409 tokens | label=Safe
  Chunk 13: 409 tokens | label=Safe
  Chunk 14: 409 tokens | label=Safe
  Chunk 15: 409 tokens | label=Safe
  Chunk 16: 409 tokens | label=Safe
  Chunk 17: 409 tokens | label=Safe
  Chunk 18: 409 tokens | label=Safe
  Chunk 19: 409 tokens | label=Safe
  Chunk 20: 409 tokens | label=Safe
  Chunk 21: 409 tokens | label=Safe
  Chunk 22: 409 tokens | label=Safe
  Chunk 23: 409 tokens | label=Safe
  Chunk 24: 409 tokens | label=Safe
  Chunk 25: 409 tokens | label=Safe
  Chunk 26:

## ✂️ 6. Chia dữ liệu & Xây dựng HuggingFace Dataset

| Tập | Tỷ lệ | Sliding Window | Prompt |
|-----|--------|----------------|--------|
| Train | 80% | ✅ Safe: sliding window · ❌ Vulnerable: giữ nguyên | 🎲 Random 1 trong 5 |
| Val | 10% | ✅ Safe: sliding window · ❌ Vulnerable: giữ nguyên | Biến thể 0 (nhất quán) |
| Test | 10% | ❌ Giữ raw | Tất cả 5 (Majority Voting) |

In [7]:
# -- Shuffle & Split 80/10/10 --
random.shuffle(raw_data)
n       = len(raw_data)
n_train = int(0.8 * n)
n_val   = int(0.9 * n)

train_raw = raw_data[:n_train]
val_raw   = raw_data[n_train:n_val]
test_raw  = raw_data[n_val:]

print(f"Train raw: {len(train_raw):,} | Val raw: {len(val_raw):,} | Test raw: {len(test_raw):,}")


def tokenize_sample(code: str, prompt_text: str, label_str: str) -> dict:
    """
    Tokenize một mẫu theo dạng BERT text-pair:
        text_a = prompt_text (instruction)
        text_b = code        (Solidity source)
    """
    encoding = tokenizer(
        prompt_text,          # text_a
        code,                 # text_b  -> tạo [CLS] a [SEP] b [SEP]
        max_length=MAX_TOKENS,
        padding="max_length",
        truncation=True,
        return_tensors=None,  # trả dict thường
    )
    encoding["labels"] = LABEL2ID[label_str]
    return encoding


def build_hf_dataset(
    raw_items: list,
    use_sliding_window: bool = False,
    random_prompt: bool = True,
    fixed_variant_idx: int = 0,
) -> Dataset:
    """
    Chuyển raw_items -> HuggingFace Dataset đã tokenize.
    Sliding window CHỈ áp dụng cho Safe (code dài).
    Vulnerable luôn được giữ nguyên (1 chunk, truncate nếu vượt budget).
    """
    records = []
    for item in raw_items:
        code_src = item["Input"]
        label    = item["Output"]
        chunks   = (
            apply_sliding_window_safe_only(tokenizer, code_src, label)
            if use_sliding_window else [(code_src, label)]
        )
        for chunk_code, chunk_label in chunks:
            v_idx       = None if random_prompt else fixed_variant_idx
            prompt_text = build_prompt_text(chunk_code, variant_idx=v_idx)
            enc         = tokenize_sample(chunk_code, prompt_text, chunk_label)
            records.append(enc)

    ds = Dataset.from_list(records)
    ds.set_format(type="torch")   # trả Tensor khi indexing
    return ds


train_dataset = build_hf_dataset(
    train_raw, use_sliding_window=True, random_prompt=True
)
val_dataset = build_hf_dataset(
    val_raw, use_sliding_window=True, random_prompt=False, fixed_variant_idx=0
)

print(f"\nTrain samples (sau sliding window Safe): {len(train_dataset):,}")
print(f"Val   samples (sau sliding window Safe): {len(val_dataset):,}")
print(f"Test  samples (raw, dùng cho MV)        : {len(test_raw):,}")
print("\n-- Ví dụ train[0] keys --")
print({k: v.shape for k, v in train_dataset[0].items()})

Train raw: 3,777 | Val raw: 472 | Test raw: 473

Train samples (sau sliding window Safe): 7,459
Val   samples (sau sliding window Safe): 1,257
Test  samples (raw, dùng cho MV)        : 473

-- Ví dụ train[0] keys --
{'input_ids': torch.Size([512]), 'attention_mask': torch.Size([512]), 'labels': torch.Size([])}


## 🤖 7. Khởi tạo Mô hình với LoRA

CodeBERT chỉ **~125M params** nên không cần quantization 4-bit.  
Ta dùng **LoRA** thuần (r=16, α=32) trên các attention layers.

| Cấu hình | CodeLlama (cũ) | CodeBERT (mới) |
|---|---|---|
| Quantization | NF4 4-bit | ❌ Không cần |
| Task type | CAUSAL_LM | SEQ_CLS |
| Target modules | q/k/v/o + MLP | query/key/value |
| Model size | 7B params | ~125M params |

In [8]:
# -- Load mô hình base với classification head --
print(f"Đang tải {MODEL_NAME} ...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    trust_remote_code=True,
)

# -- Cấu hình LoRA cho Sequence Classification --
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # RoBERTa/BERT attention projection names
    target_modules=["query", "key", "value"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,   # Thay CAUSAL_LM -> SEQ_CLS
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Đảm bảo classification head được train
for name, param in model.named_parameters():
    if "classifier" in name:
        param.requires_grad = True

print("✅ Mô hình CodeBERT + LoRA đã sẵn sàng!")

Đang tải microsoft/codebert-base ...


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

trainable params: 1,476,866 || all params: 126,124,036 || trainable%: 1.1710
✅ Mô hình CodeBERT + LoRA đã sẵn sàng!


## 🏋️ 8. Huấn luyện với HuggingFace Trainer

Dùng `Trainer` chuẩn (thay vì `SFTTrainer` của TRL vốn dành cho causal LM).  
Đánh giá định kỳ mỗi **200 steps**, giữ **best checkpoint** theo F1.

In [9]:
import os, glob

def compute_metrics(eval_pred):
    """Tính F1 class Vulnerable trong quá trình training."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, pos_label=LABEL2ID["Vulnerable"], average="binary")
    return {"f1_vulnerable": f1}


OUTPUT_DIR = "./codebert-vuln-lora"

# =============================================================
# Tự động tìm checkpoint gần nhất để resume
# =============================================================
def find_last_checkpoint(output_dir: str):
    """
    Tìm checkpoint có số step lớn nhất trong output_dir.
    Trả về đường dẫn checkpoint hoặc None nếu chưa có.
    """
    ckpt_dirs = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    if not ckpt_dirs:
        return None
    # Sắp xếp theo số step (checkpoint-200, checkpoint-400, ...)
    ckpt_dirs = sorted(ckpt_dirs, key=lambda x: int(x.split("-")[-1]))
    return ckpt_dirs[-1]  # checkpoint mới nhất


last_checkpoint = find_last_checkpoint(OUTPUT_DIR)
if last_checkpoint:
    print(f"🔁 Tìm thấy checkpoint: {last_checkpoint}")
    print(f"   → Sẽ resume training từ checkpoint này.")
else:
    print("🆕 Không có checkpoint. Bắt đầu training từ đầu.")


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,          # Giữ 3 checkpoint gần nhất (tăng từ 2 để an toàn hơn)
    load_best_model_at_end=True,
    metric_for_best_model="f1_vulnerable",
    greater_is_better=True,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    report_to="none",
    optim="adamw_torch",
    dataloader_pin_memory=False,  # Tắt pin_memory khi không có GPU accelerator
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\n🚀 Bắt đầu huấn luyện...")
trainer.train(resume_from_checkpoint=last_checkpoint)  # None = train từ đầu, path = resume
print("\n✅ Huấn luyện hoàn tất!")

🆕 Không có checkpoint. Bắt đầu training từ đầu.

🚀 Bắt đầu huấn luyện...


`use_return_dict` is deprecated! Use `return_dict` instead!
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,F1 Vulnerable
200,0.899260,0.298317,0.737401
400,0.655821,0.288942,0.824034
600,0.601887,0.207997,0.862069
800,0.468054,0.225523,0.871560
1000,0.342652,0.218919,0.881279
1170,0.359362,0.215799,0.873563


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector


✅ Huấn luyện hoàn tất!


## 💾 9. Lưu Adapter LoRA

In [10]:
SAVE_PATH = "./codebert-vuln-lora-final"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Adapter LoRA đã lưu tại: {SAVE_PATH}")

✅ Adapter LoRA đã lưu tại: ./codebert-vuln-lora-final


## 🗳️ 10. Đánh giá Majority Voting trên tập Test

**Quy trình cho mỗi mẫu test:**
1. Chạy mô hình qua **tất cả 5 biến thể prompt** của cùng đoạn code
2. Thu thập 5 dự đoán (logits -> argmax -> label)
3. Nếu **≥ 3 dự đoán là "Vulnerable"** → nhãn cuối = `Vulnerable`, ngược lại → `Safe`
4. Tính **F1-score** và **Classification Report** từ `sklearn`

> **Điểm khác biệt so với CodeLlama:** Không cần generate text → chỉ cần forward pass + argmax.  
> Nhanh hơn ~10-20x và không có vấn đề parsing output (Vulnerable/Safe parse từ text).

In [11]:
# =============================================================
# Hàm dự đoán cho 1 mẫu với 1 biến thể prompt
# =============================================================

def predict_single(
    model,
    tokenizer,
    code: str,
    variant_idx: int,
) -> str:
    """
    Phân loại 1 đoạn code với biến thể prompt xác định.
    Trả về "Vulnerable" hoặc "Safe" (không cần parse text).
    """
    prompt_text = build_prompt_text(code, variant_idx=variant_idx)
    encoding = tokenizer(
        prompt_text,
        code,
        max_length=MAX_TOKENS,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    encoding = {k: v.to(model.device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**encoding)

    pred_id = torch.argmax(outputs.logits, dim=-1).item()
    return ID2LABEL[pred_id]


# =============================================================
# Hàm đánh giá Majority Voting (chạy toàn bộ tập test)
# =============================================================

def evaluate_majority_voting(
    model,
    tokenizer,
    test_raw: list,
) -> tuple:
    """
    Chạy Majority Voting (5 biến thể prompt) trên tập test.
    Với mỗi mẫu: lặp qua cả 5 biến thể, vote >= 3 Vulnerable -> Vulnerable.

    Returns:
        (y_true: list[str], y_pred: list[str])
    """
    model.eval()
    y_true, y_pred = [], []

    print(f"Tổng mẫu test: {len(test_raw):,}")
    for i, item in enumerate(test_raw):
        code_src   = item["Input"]
        true_label = item["Output"]

        # Lặp qua 5 biến thể prompt (KHÔNG random)
        votes = []
        for v_idx in range(5):
            prediction = predict_single(model, tokenizer, code_src, variant_idx=v_idx)
            votes.append(prediction)

        # Majority Voting: >= 3 "Vulnerable" -> Vulnerable
        n_vuln      = sum(1 for v in votes if v == "Vulnerable")
        final_label = "Vulnerable" if n_vuln >= 3 else "Safe"

        y_true.append(true_label)
        y_pred.append(final_label)

        # Log mỗi 20 mẫu hoặc mẫu cuối
        if (i + 1) % 20 == 0 or (i + 1) == len(test_raw):
            correct = "✅" if final_label == true_label else "❌"
            print(
                f"  [{i+1:5d}/{len(test_raw)}] "
                f"votes_vuln={n_vuln}/5 -> pred={final_label} "
                f"| true={true_label} {correct}"
            )

    return y_true, y_pred


# =============================================================
# Chạy đánh giá & In kết quả
# =============================================================

print("🔍 Bắt đầu đánh giá Majority Voting trên tập test...")
y_true, y_pred = evaluate_majority_voting(model, tokenizer, test_raw)

# F1-score (class Vulnerable là positive)
f1 = f1_score(y_true, y_pred, pos_label="Vulnerable", average="binary")

print(f"\n{'='*60}")
print(f"  📊  F1-Score (class Vulnerable): {f1:.4f}")
print(f"{'='*60}")
print("\n📋 Classification Report (tập Test):")
print(classification_report(
    y_true, y_pred,
    target_names=["Safe", "Vulnerable"],
    digits=4,
))

🔍 Bắt đầu đánh giá Majority Voting trên tập test...
Tổng mẫu test: 473
  [   20/473] votes_vuln=0/5 -> pred=Safe | true=Safe ✅
  [   40/473] votes_vuln=0/5 -> pred=Safe | true=Safe ✅
  [   60/473] votes_vuln=0/5 -> pred=Safe | true=Safe ✅
  [   80/473] votes_vuln=0/5 -> pred=Safe | true=Safe ✅
  [  100/473] votes_vuln=5/5 -> pred=Vulnerable | true=Vulnerable ✅
  [  120/473] votes_vuln=5/5 -> pred=Vulnerable | true=Vulnerable ✅
  [  140/473] votes_vuln=0/5 -> pred=Safe | true=Safe ✅
  [  160/473] votes_vuln=0/5 -> pred=Safe | true=Safe ✅
  [  180/473] votes_vuln=5/5 -> pred=Vulnerable | true=Vulnerable ✅
  [  200/473] votes_vuln=5/5 -> pred=Vulnerable | true=Vulnerable ✅
  [  220/473] votes_vuln=5/5 -> pred=Vulnerable | true=Vulnerable ✅
  [  240/473] votes_vuln=5/5 -> pred=Vulnerable | true=Vulnerable ✅
  [  260/473] votes_vuln=0/5 -> pred=Safe | true=Safe ✅
  [  280/473] votes_vuln=5/5 -> pred=Vulnerable | true=Vulnerable ✅
  [  300/473] votes_vuln=5/5 -> pred=Vulnerable | true=Safe ❌